In [ ]:
# Install required libraries
!pip install -q kagglehub pandas scikit-learn seaborn matplotlib

import kagglehub
import pandas as pd
import os
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import joblib
import seaborn as sns
import matplotlib.pyplot as plt

print("Initializing RiskSight ML Pipeline...")
print("Step 1: Downloading dataset via kagglehub...")

# 1. Download the dataset using the code from your Kaggle screenshot
path = kagglehub.dataset_download("stacknishant/nse-1800-stocks-historical-yearly-financial-ratios")
print("Dataset downloaded to:", path)

# 2. Find the CSV file inside the downloaded folder
csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]

if not csv_files:
    print("❌ Error: No CSV file found in the downloaded folder.")
else:
    # 3. Load the actual CSV into Pandas
    csv_filename = csv_files[0] # Grab the first CSV it finds
    full_csv_path = os.path.join(path, csv_filename)

    print(f"\nLoading data from {csv_filename}...")
    df = pd.read_csv(full_csv_path)

    print(f"✅ Successfully loaded real Indian market data for {len(df)} records.")

    # 4. Preview the actual column names so we can map them to our ML logic
    print("\n=========================================")
    print("        EXACT COLUMNS IN DATASET         ")
    print("=========================================\n")
    for col in df.columns.tolist():
        print(f"- {col}")

    display(df.head())

Initializing RiskSight ML Pipeline...
Step 1: Downloading dataset via kagglehub...


100%|██████████| 8.70M/8.70M [00:00<00:00, 87.5MB/s]

Extracting files...


Dataset downloaded to: /root/.cache/kagglehub/datasets/stacknishant/nse-1800-stocks-historical-yearly-financial-ratios/versions/1

Loading data from nsestockhistoricalratios.csv...
✅ Successfully loaded real Indian market data for 19767 records.

        EXACT COLUMNS IN DATASET         

- Year
- Stock
- Asset Turnover
- Average Inventory
- Average Payables
- Average Receivables
- Book Value Per Share
- Capex Per Share
- Capex To Depreciation
- Capex To Operating Cash Flow
- Capex To Revenue
- Capital Expenditure Coverage Ratio
- Cash Conversion Cycle
- Cash Flow Coverage Ratios
- Cash Flow To Debt Ratio
- Cash Per Share
- Cash Ratio
- Company Equity Multiplier
- Current Ratio
- Days Of Inventory On Hand
- Days Of Inventory Outstanding
- Days Of Payables Outstanding
- Days Of Sales Outstanding
- Days Payables Outstanding
- Days Sales Outstanding
- Debt Equity Ratio
- Debt Ratio
- Debt To Assets
- Debt To Equity
- Dividend Paid And Capex Coverage Ratio
- Dividend Payout Ratio
- Dividen

,Year,Stock,Asset Turnover,Average Inventory,Average Payables,Average Receivables,Book Value Per Share,Capex Per Share,Capex To Depreciation,Capex To Operating Cash Flow,...,Return On Tangible Assets,Revenue Per Share,Sales General And Administrative To Revenue,Shareholders Equity Per Share,Short Term Coverage Ratios,Stock Based Compensation To Revenue,Tangible Asset Value,Tangible Book Value Per Share,Total Debt To Capitalization,Working Capital
0,2012,20MICRONS,1.067539,5.007655e+08,1.641265e+08,6.477285e+08,21.841254,-13.640144,-5.126600,-0.943816,...,0.016394,99.063718,0.000000,21.841254,0.644230,0.0,6.689560e+08,21.866836,0.675178,-1.752910e+08
1,2012,21STCENMGM,0.005456,0.000000e+00,1.441035e+08,0.000000e+00,45.069524,-0.000476,-0.001038,-0.000528,...,-0.083987,0.359048,0.000000,45.069524,17.229091,0.0,4.672300e+08,45.069524,0.001161,2.584020e+08
2,2012,360ONE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2012,3IINFOLTD,0.347382,1.360000e+07,2.986400e+09,4.109800e+09,208.086809,-9.683324,-0.184386,-0.860146,...,-0.445874,298.362874,0.000000,208.086809,0.437373,0.0,-1.722340e+10,209.342729,0.694964,-2.986200e+09
4,2012,3MINDIA,1.479883,2.194638e+09,1.372214e+09,1.169509e+09,582.718439,-125.238902,-3.886570,-2.212066,...,0.049254,1396.099802,0.011906,582.718439,0.384143,0.0,6.548382e+09,581.299717,0.209261,2.211572e+09


In [ ]:
print("Step 2: Cleaning Data and Engineering 'Distress' Labels...")

# 1. Clean the Data
# We drop rows that are completely missing our core metrics
core_metrics = ['Debt To Equity', 'Current Ratio', 'Interest Coverage', 'Return On Capital Employed']
df_clean = df.dropna(subset=core_metrics).copy()

print(f"Removed {len(df) - len(df_clean)} rows with missing core metrics.")

# 2. Engineer the Distress Label
def assign_distress_label(row):
    risk_score = 0

    # Flag 1: High Leverage (Debt > 2.5x Equity)
    if row['Debt To Equity'] > 2.5:
        risk_score += 1

    # Flag 2: Poor Liquidity (Cannot cover short term liabilities)
    if row['Current Ratio'] < 0.8:
        risk_score += 1

    # Flag 3: Severe Solvency Issue (Operating profit doesn't cover interest)
    if row['Interest Coverage'] < 1.0:
        risk_score += 2  # Higher weight for missing interest payments

    # Flag 4: Capital Destruction
    if row['Return On Capital Employed'] < 0:
        risk_score += 1

    # If the combined risk score hits 3 or more, the company is flagged as distressed
    return 1 if risk_score >= 3 else 0

# Apply the logic
df_clean['Is_Distressed'] = df_clean.apply(assign_distress_label, axis=1)

distress_count = df_clean['Is_Distressed'].sum()
print(f"✅ Labeling complete.")
print(f"📊 Market Breakdown: {len(df_clean) - distress_count} Healthy | {distress_count} Distressed")

Step 2: Cleaning Data and Engineering 'Distress' Labels...
Removed 1281 rows with missing core metrics.
✅ Labeling complete.
📊 Market Breakdown: 14661 Healthy | 3825 Distressed


In [ ]:
from google.colab import files

# Save the cleaned and labeled dataframe to a CSV file
csv_filename = "cleaned_nse_financials_labeled.csv"
df_clean.to_csv(csv_filename, index=False)

print(f"✅ Cleaned data saved as {csv_filename}")

# Trigger the download to your local computer
files.download(csv_filename)

✅ Cleaned data saved as cleaned_nse_financials_labeled.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
print("Step 3: Training the Supervised Random Forest Brain...")

# 1. Define Features (X) and Target (y)
# We drop 'Stock', 'Year', and our target label so the model only looks at the math
X = df_clean.drop(['Stock', 'Year', 'Is_Distressed'], axis=1)
y = df_clean['Is_Distressed']

# 2. Handle any remaining NaNs in the dataset by filling them with the median of their column
X = X.fillna(X.median())

# 3. Train/Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training on {len(X_train)} records, validating on {len(X_test)} records...")

# 4. Train the Random Forest
rf_classifier = RandomForestClassifier(
    n_estimators=150,
    max_depth=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf_classifier.fit(X_train, y_train)

# 5. Evaluate the Model
y_pred = rf_classifier.predict(X_test)

print("\n====================================================")
print("             RISKSIGHT MODEL EVALUATION             ")
print("====================================================\n")
print(classification_report(y_test, y_pred, target_names=['Healthy (0)', 'Distressed (1)']))

# 6. Save the Model
model_filename = 'risksight_nse_rf_model.pkl'
joblib.dump(rf_classifier, model_filename)
print(f"\n💾 Pipeline Output: Model successfully serialized and saved as '{model_filename}'")

Step 3: Training the Supervised Random Forest Brain...
Training on 14788 records, validating on 3698 records...

             RISKSIGHT MODEL EVALUATION             

                precision    recall  f1-score   support

   Healthy (0)       1.00      1.00      1.00      2933
Distressed (1)       0.99      0.99      0.99       765

      accuracy                           1.00      3698
     macro avg       0.99      0.99      0.99      3698
  weighted avg       1.00      1.00      1.00      3698


💾 Pipeline Output: Model successfully serialized and saved as 'risksight_nse_rf_model.pkl'
